# ⛏️ Servidor Minecraft **Fabric 26.3** en Google Colab (no premium)

Basado en MineColab, rehecho para versiones nuevas:

- **Java 25** (Temurin), que es lo que pide Minecraft 26.x.
- **Fabric** bajado directo de `meta.fabricmc.net` (serverjars.com ya no existe).
- **online-mode=false**: entran cuentas no premium (TLauncher, SKLauncher, etc.).
- Túnel **bore** por defecto: gratis, sin cuenta. También `playit` (dirección fija, pide cuenta gratis) o `ngrok` (TCP pide tarjeta verificada).
- El servidor corre en el disco local de Colab (rápido) y se **guarda en Google Drive** cada 10 minutos y al apagarlo con `stop`.

**Pasos:** ejecutá las celdas 1, 2 y 3 en orden. La celda 3 es la consola: escribí comandos ahí (`op TuNick`, `stop`...). Apagá siempre con `stop` para que guarde en Drive.

> ⚠️ Con online-mode apagado cualquiera puede entrar con cualquier nick (incluso el tuyo). Si compartís la IP, considerá un mod de login (p. ej. *EasyAuth*) o usar `whitelist`.


In [ ]:
# @title 1 · Configuración { display-mode: "form" }
VERSION = "26.3"  # @param {type:"string"}
CARPETA_DRIVE = "Minecraft-Fabric-26.3"  # @param {type:"string"}
USAR_DRIVE = True  # @param {type:"boolean"}
TUNEL = "bore"  # @param ["bore", "playit", "ngrok"]
NGROK_TOKEN = ""  # @param {type:"string"}
NGROK_REGION = "sa"  # @param ["sa", "us", "eu", "ap", "au", "jp", "in"]
RAM_GB = 0  # @param {type:"integer"}
ONLINE_MODE = False  # @param {type:"boolean"}
INSTALAR_FABRIC_API = True  # @param {type:"boolean"}
MOTD = "Servidor Fabric 26.3 en Colab"  # @param {type:"string"}
MAX_JUGADORES = 10  # @param {type:"integer"}
# RAM_GB = 0 -> usa ~75% de la RAM de la máquina.


In [ ]:
# @title 2 · Instalar / actualizar { display-mode: "form" }
import os, json, glob, shutil, subprocess, tarfile, urllib.request, urllib.parse

def log(t): print(f"\033[1;32m[LOG]\033[0m {t}", flush=True)
def sh(cmd): subprocess.run(cmd, shell=True, check=True)
def get_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "minecolab-fabric"})
    return json.load(urllib.request.urlopen(req))
def bajar(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "minecolab-fabric"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)

LOCAL = "/content/server" if os.path.isdir("/content") else os.path.abspath("server")
DRIVE = None
if USAR_DRIVE:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE = os.path.join("/content/drive/MyDrive", CARPETA_DRIVE)
    os.makedirs(DRIVE, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

# Traer lo guardado en Drive (mundo, mods, configs)
if DRIVE and os.listdir(DRIVE):
    log(f"Copiando servidor desde Drive ({DRIVE})...")
    sh(f'rsync -a "{DRIVE}/" "{LOCAL}/"')

# Java 25
JAVA_DIR = os.path.join(os.path.dirname(LOCAL), "java25")
JAVA = os.path.join(JAVA_DIR, "bin", "java")
if not os.path.exists(JAVA):
    log("Instalando Java 25 (Temurin JRE)...")
    tgz = JAVA_DIR + ".tar.gz"
    bajar("https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jre/hotspot/normal/eclipse", tgz)
    os.makedirs(JAVA_DIR, exist_ok=True)
    sh(f'tar -xzf "{tgz}" -C "{JAVA_DIR}" --strip-components=1 && rm "{tgz}"')
log(subprocess.run([JAVA, "-version"], capture_output=True, text=True).stderr.splitlines()[0])

# Fabric server launcher (descarga el server de Mojang y las libs en el primer arranque)
loaders = get_json(f"https://meta.fabricmc.net/v2/versions/loader/{VERSION}")
if not loaders:
    raise SystemExit(f"Fabric no soporta todavía la versión {VERSION}")
loader = next((l["loader"]["version"] for l in loaders if l["loader"]["stable"]), loaders[0]["loader"]["version"])
installer = get_json("https://meta.fabricmc.net/v2/versions/installer")[0]["version"]
jar = f"fabric-server-mc.{VERSION}-loader.{loader}.jar"
if not os.path.exists(os.path.join(LOCAL, jar)):
    for viejo in glob.glob(os.path.join(LOCAL, "fabric-server-mc.*.jar")):
        os.remove(viejo)
    log(f"Bajando Fabric loader {loader} para {VERSION}...")
    bajar(f"https://meta.fabricmc.net/v2/versions/loader/{VERSION}/{loader}/{installer}/server/jar", os.path.join(LOCAL, jar))

# Fabric API (casi todos los mods la necesitan)
os.makedirs(os.path.join(LOCAL, "mods"), exist_ok=True)
if INSTALAR_FABRIC_API:
    q = urllib.parse.urlencode({"game_versions": json.dumps([VERSION]), "loaders": json.dumps(["fabric"])})
    vers = get_json(f"https://api.modrinth.com/v2/project/fabric-api/version?{q}")
    if vers:
        f = vers[0]["files"][0]
        if not os.path.exists(os.path.join(LOCAL, "mods", f["filename"])):
            for viejo in glob.glob(os.path.join(LOCAL, "mods", "fabric-api-*.jar")):
                os.remove(viejo)
            log(f"Bajando {f['filename']}...")
            bajar(f["url"], os.path.join(LOCAL, "mods", f["filename"]))
    else:
        log("No hay Fabric API para esta versión todavía, sigo sin ella.")

# EULA + server.properties
open(os.path.join(LOCAL, "eula.txt"), "w").write("eula=true\n")
props_path = os.path.join(LOCAL, "server.properties")
props = {}
if os.path.exists(props_path):
    for line in open(props_path, encoding="utf-8"):
        if "=" in line and not line.startswith("#"):
            k, v = line.rstrip("\n").split("=", 1)
            props[k] = v
props.update({
    "online-mode": str(ONLINE_MODE).lower(),
    "enforce-secure-profile": "false",  # que los no premium puedan chatear
    "white-list": "false",  # con whitelist los nicks no premium no matchean el UUID
    "motd": MOTD,
    "max-players": str(MAX_JUGADORES),
    "server-port": "25565",
})
with open(props_path, "w", encoding="utf-8") as f:
    f.writelines(f"{k}={v}\n" for k, v in props.items())

json.dump({"jar": jar, "java": JAVA}, open(os.path.join(LOCAL, "colab.json"), "w"))
if DRIVE:
    sh(f'rsync -a "{LOCAL}/" "{DRIVE}/"')
log(f"Listo: {jar} en {LOCAL}" + (f" (copia en Drive: {DRIVE})" if DRIVE else ""))
log("Mods: poné los .jar en la carpeta mods/ de Drive y volvé a correr esta celda.")


In [ ]:
# @title 3 · Iniciar servidor (esta celda es la consola) { display-mode: "form" }
import os, re, json, time, threading, subprocess, platform

cfg = json.load(open(os.path.join(LOCAL, "colab.json")))
BIN = os.path.join(os.path.dirname(LOCAL), "bin")
os.makedirs(BIN, exist_ok=True)

def iniciar_tunel():
    """Arranca el túnel en segundo plano y devuelve la dirección pública (o None)."""
    if TUNEL == "bore":
        bore = os.path.join(BIN, "bore")
        if not os.path.exists(bore):
            tag = get_json("https://api.github.com/repos/ekzhang/bore/releases/latest")["tag_name"]
            bajar(f"https://github.com/ekzhang/bore/releases/download/{tag}/bore-{tag}-x86_64-unknown-linux-musl.tar.gz", bore + ".tgz")
            sh(f'tar -xzf "{bore}.tgz" -C "{BIN}" && rm "{bore}.tgz" && chmod +x "{bore}"')
        p = subprocess.Popen([bore, "local", "25565", "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r"listening at bore\.pub:(\d+)", re.sub(r"\x1b\[[0-9;]*m", "", line))
            if m:
                threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()
                return f"bore.pub:{m.group(1)}"
        return None
    if TUNEL == "playit":
        playit = os.path.join(BIN, "playit")
        if not os.path.exists(playit):
            bajar("https://github.com/playit-cloud/playit-agent/releases/latest/download/playit-linux-amd64", playit)
            os.chmod(playit, 0o755)
        secret = os.path.join(DRIVE or LOCAL, "playit.toml")
        p = subprocess.Popen([playit, "--secret_path", secret, "--stdout"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        def leer():
            for line in p.stdout:
                if "claim" in line.lower() or "playit.gg" in line or "error" in line.lower():
                    print("[playit]", line.rstrip(), flush=True)
        threading.Thread(target=leer, daemon=True).start()
        log("playit: la primera vez abrí el link de 'claim' que aparece arriba, iniciá sesión y creá un túnel 'Minecraft Java' al puerto 25565. La dirección queda fija en tu panel de playit.gg.")
        return None
    if TUNEL == "ngrok":
        try:
            from pyngrok import ngrok, conf
        except ImportError:
            sh("pip install -q pyngrok")
            from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN
        conf.get_default().region = NGROK_REGION
        return ngrok.connect(25565, "tcp").public_url.replace("tcp://", "")

direccion = iniciar_tunel()
if direccion:
    print(f"\n\033[1;36m==> IP del servidor: {direccion}\033[0m  (cambia cada vez que lo reinicies)\n", flush=True)

ram = RAM_GB or max(2, int(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30 * 0.75))
flags = f"-Xms{ram}G -Xmx{ram}G -XX:+UseG1GC -XX:+ParallelRefProcEnabled -XX:MaxGCPauseMillis=200 -XX:+UnlockExperimentalVMOptions -XX:+DisableExplicitGC -XX:+AlwaysPreTouch -XX:G1NewSizePercent=30 -XX:G1MaxNewSizePercent=40 -XX:G1HeapRegionSize=8M -XX:G1ReservePercent=20 -XX:InitiatingHeapOccupancyPercent=15"
log(f"Iniciando servidor con {ram} GB de RAM...")
proc = subprocess.Popen([cfg["java"], *flags.split(), "-jar", cfg["jar"], "nogui"], cwd=LOCAL,
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lock = threading.Lock()

def enviar(cmd):
    with lock:
        proc.stdin.write(cmd + "\n"); proc.stdin.flush()

def sync_drive():
    if DRIVE:
        subprocess.run(f'rsync -a --delete --exclude=libraries --exclude=versions --exclude=.fabric "{LOCAL}/" "{DRIVE}/"', shell=True)

def salida():
    for line in proc.stdout:
        print(line, end="", flush=True)
threading.Thread(target=salida, daemon=True).start()

def autoguardado():
    while proc.poll() is None:
        time.sleep(600)
        if proc.poll() is not None or not DRIVE:
            break
        enviar("save-off"); enviar("save-all flush"); time.sleep(15)
        sync_drive(); enviar("save-on")
        log("Mundo guardado en Drive.")
threading.Thread(target=autoguardado, daemon=True).start()

try:
    while proc.poll() is None:
        try:
            cmd = input()
        except EOFError:
            break
        if cmd.strip() and proc.poll() is None:
            enviar(cmd.strip())
        if cmd.strip() == "stop":
            break
except KeyboardInterrupt:
    log("Interrumpido: apagando servidor...")
    if proc.poll() is None:
        enviar("stop")
proc.wait()
log("Servidor apagado. Guardando en Drive...")
sync_drive()
log("Listo.")
